In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
import random
import os

from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx

os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [7]:
batch = 1
cur_channel = 2
cur_size = 3
cur_nodes = cur_channel * cur_size**2

k = 2
s = 2

tensor_2d = torch.arange(cur_nodes).reshape(1,cur_channel,cur_size,cur_size).float()
print(tensor_2d)
indices = F.unfold(tensor_2d, (k,k), stride = 1, padding= 1)

# i_unf = indices.view(1, cur_channel, k*k, -1)
# i_unf = indices.view(cur_channel, k*k, -1).int()

# # edges = i_unf.transpose(2,3)
# # pool_ind = edges.reshape(1, edges.shape[1]*edges.shape[2], edges.shape[3])
# # print(i_unf)
# print(i_unf)

# print()

cnn_indices = F.unfold(tensor_2d, (k,k), stride = s, padding=1).transpose(1,2).int()
print(cnn_indices)
print(cnn_indices.shape)

cnn_indices = F.unfold(tensor_2d, (k,k), stride = s, padding=0).transpose(1,2).int()
print(cnn_indices)
print(cnn_indices.shape)

tensor([[[[ 0.,  1.,  2.],
          [ 3.,  4.,  5.],
          [ 6.,  7.,  8.]],

         [[ 9., 10., 11.],
          [12., 13., 14.],
          [15., 16., 17.]]]])
tensor([[[ 0,  0,  0,  0,  0,  0,  0,  9],
         [ 0,  0,  1,  2,  0,  0, 10, 11],
         [ 0,  3,  0,  6,  0, 12,  0, 15],
         [ 4,  5,  7,  8, 13, 14, 16, 17]]], dtype=torch.int32)
torch.Size([1, 4, 8])
tensor([[[ 0,  1,  3,  4,  9, 10, 12, 13]]], dtype=torch.int32)
torch.Size([1, 1, 8])


In [66]:
shape = [1, 4, 8]
array = np.arange(np.prod(shape)).reshape(shape)

# Flatten the array
flat_index = 9  # 10th element corresponds to index 9 in zero-based indexing

# Convert the flat index to a multi-dimensional index
multi_dim_index = np.unravel_index(flat_index, shape)

print(multi_dim_index)

(0, 1, 1)


In [57]:
ori = torch.zeros((2,2,3,3))
k1 = torch.zeros((3, 2, 2, 2))


k1[0,0,0,0] = 1.
k1[0,0,1,1] = 2.
k1[1,0,1,1] = 3.
k1[2,0,0,1] = 4.
k1[1,1,1,1] = 2.
k1[2,1,1,0] = 5.


ori[0,0,0,1] = 1.
ori[0,1,1,2] = 2.
# ori[0,1,1,4] = 4.
ori[0,0,1,2] = 12.
ori[0,0,2,2] = 1.
ori[0,1,2,2] = 11.
# ori[0,0,3,4] = 7.
ori[0,0,2,1] = 9.
# ori[0,1,4,3] = 1.
# ori[0,1,4,0] = 11.
ori[0,1,0,1] = 14.

# print(ori)
# print()
# print(k1)
# print()
k = k1.view(k1.size(0),-1).T

out_size = (ori.shape[2] - k1.shape[2])//1 +1

ori_unf = F.unfold(ori,(k1.shape[2],k1.shape[3]), stride=1)
# print(ori_unf.shape)

# print(ori_unf.transpose(1,2))
# print()

batch = 2
l2_channel = 3
map_size = out_size**2

reO = ori_unf.transpose(1,2)

w = k.unsqueeze(0).transpose(2, 0)
print(w.shape)

mask = torch.ones((l2_channel, reO.shape[1], reO.shape[2]))
res = None
for i in range(l2_channel):
    y = (mask[i]*reO).transpose(1,2)
    edges = (y.unsqueeze(1) * w[None,i,:,:]).transpose(2,3)
    print(edges.shape)
    res = edges if res == None else torch.cat((res, edges), axis=1)

edges1 = ori_unf.unsqueeze(1) * w[None,:,:,:]
edges1 = edges1.transpose(2,3)
print(edges1.shape)

print(res == edges1)
        
mask[0, 1, 1] = 0

res = None
for i in range(l2_channel):
    y = (mask[i]*reO)
    # print(y.shape)
    res = y if res == None else torch.cat((res, y), axis=1)

# print(res.shape)

out1 = (reO @ (k1.view(k1.size(0),-1).t())).transpose(1,2)


y = F.fold(out1, (out_size,out_size), (1,1))



torch.Size([3, 8, 1])
torch.Size([2, 1, 4, 8])
torch.Size([2, 1, 4, 8])
torch.Size([2, 1, 4, 8])
torch.Size([2, 3, 4, 8])
tensor([[[[True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True]],

         [[True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True]],

         [[True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True]]],


        [[[True, True, True, True, True, True, True, True],
          [True, True, True, True, True, True, True, True],
          [True, True, True, T

In [45]:
k_size = 2
stride = 2

out = y
# print(y)

batch = out.shape[0]
channel = out.shape[1]

out_s0 = (out.shape[2] - k_size) // stride + 1
out_s1 = (out.shape[3] - k_size) // stride + 1

i_unf = F.unfold(out, (k_size, k_size), stride=2) 

# print(i_unf.transpose(1,2))

i_unf = i_unf.view(batch, channel, k_size*k_size, -1)
# print(i_unf)

batch = 2
l2_channel = 3
map_size = out_size**2

i_unf1 = i_unf.view(batch*channel, k_size*k_size, -1)
# print(i_unf1)

mask = torch.ones((l2_channel, i_unf1.shape[1], i_unf1.shape[2]))
print(mask.shape)

res = None
for i in range(l2_channel):
    y1 = mask[i] * i_unf[:,i,:,:].unsqueeze(1)
    res = y1 if res == None else torch.cat((res, y1), axis=1)

# print(i_unf1)
print(res.shape)
res = res.view(batch*channel, k_size*k_size, -1)

x = torch.max(i_unf1, dim = 1, keepdim=True).values

x1 = torch.max(res, dim = 1, keepdim=True).values
print(x == x1)
x = x.view(batch, channel, out_s0, out_s1)



torch.Size([3, 4, 1])
torch.Size([2, 3, 4, 1])
tensor([[[True]],

        [[True]],

        [[True]],

        [[True]],

        [[True]],

        [[True]]])


In [75]:
nodes_num = 4*4*3+3*2*2

model_dims = {
    1: {"name": "cnn", "dim": {"channel": 3, "kernel": 2, "stride": 1, "out_size": 4}},
    2: {"name": "pooling", "dim": {"channel": 3, "kernel": 2, "stride": 2, "out_size": 2}},
}

In [76]:
w_avg = edge_v2.squeeze()
print(w_avg)

tensor([ 0., 25.,  0.,  2.,  0.,  0., 12.,  0., 18.,  4.,  0.,  9.,  1., 14.,
         2.,  0.,  0., 36.,  0., 25.,  0.,  8.,  0.,  0., 27.,  6.,  0.,  0.,
         0., 21.,  2.,  0.,  4.,  0.,  0., 48.,  0.,  0., 55.,  0.,  0.,  4.,
        91.,  8.,  0.,  0.,  0., 33.])


In [80]:
# build adjacent matrix
adjacent_m = np.zeros((nodes_num, nodes_num), dtype=np.float32)

current_l = 1
start_col = 0
cur_s_col = 0
nodes_total = 0

layer_num = len(model_dims)

for i in range(1, layer_num):
    current_l = i
    next_l= i + 1
    
    cur_name = model_dims[current_l]["name"]
    cur_dim = model_dims[current_l]["dim"]
    cur_size = cur_dim['out_size']
    
    cur_channel = 1 if (cur_name == "fc") else cur_dim["channel"]
    
    if cur_name == "input":
        cur_nodes = cur_channel * cur_size**2
    else:
        cur_nodes = cur_size if (cur_name == "fc") else cur_dim["channel"]*(cur_size**2)

    nxt_name = model_dims[next_l]["name"]
    nxt_dim = model_dims[next_l]["dim"]
    out_size = nxt_dim["out_size"]
    
    # cnn layer
    if (nxt_name == "cnn" or nxt_name == "pooling"):
        k = nxt_dim["kernel"]
        s = nxt_dim["stride"]
        c = nxt_dim["channel"]
        
        step = k**2      
        edges_n = out_size**2 * c * cur_channel * step
        n = 0
        tensor_2d = torch.arange(cur_nodes).reshape(1,cur_channel,cur_size,cur_size).float()
        
        if (nxt_name == "cnn"):
            indices = F.unfold(tensor_2d, (k,k), stride = s).transpose(1,2).int()
            end_col = start_col + step*cur_channel
            for ur_c in range(c): 
                for l in range(indices.shape[1]):
                    cur_idx = indices[0,l].tolist()  

                    assert(len(cur_idx) == step*cur_channel)
                    adjacent_m[cur_idx, cur_nodes+n] = w_avg[start_col : end_col]
        
                    start_col = end_col
                    end_col = start_col + step*cur_channel
                    n += 1
        else:
            indices = F.unfold(tensor_2d, (k,k), stride = s)
            i_unf = indices.view(1, cur_channel, k*k, -1).transpose(2,3)
            indices = i_unf.reshape(i_unf.shape[0], i_unf.shape[1]*i_unf.shape[2], i_unf.shape[3]).int()
            
            for l in range(indices.shape[1]):
                end_col = start_col + step
                cur_idx = indices[0,l].tolist()
                assert(len(cur_idx) == step)
                adjacent_m[cur_idx, cur_nodes+n] = w_avg[start_col : end_col]

                start_col = end_col
                end_col = start_col + step
                n += 1
                
        nodes_total += cur_nodes
        # print(start_col)  
    
    # fc layer
    elif (nxt_name == "fc"):  
        cur_s_col = nodes_total + cur_size
        cur_e_col = nodes_total + cur_size + out_size

        end_col = start_col + out_size

        for node in range(nodes_total, nodes_total + cur_nodes, 1):
            # print(f'i : {node}, start col : {cur_s_col}, end_col : {cur_e_col}, from {start_col} to {end_col}')
            adjacent_m[node, cur_s_col : cur_e_col] = w_avg[start_col : end_col]        
            
            start_col = end_col
            end_col = end_col + out_size            
                
        nodes_total += cur_nodes
        
        

[0, 1, 4, 5]
48
[ 0. 25.  0.  2.]
[2, 3, 6, 7]
49
[ 0.  0. 12.  0.]
[8, 9, 12, 13]
50
[18.  4.  0.  9.]
[10, 11, 14, 15]
51
[ 1. 14.  2.  0.]
[16, 17, 20, 21]
52
[ 0. 36.  0. 25.]
[18, 19, 22, 23]
53
[0. 8. 0. 0.]
[24, 25, 28, 29]
54
[27.  6.  0.  0.]
[26, 27, 30, 31]
55
[ 0. 21.  2.  0.]
[32, 33, 36, 37]
56
[ 4.  0.  0. 48.]
[34, 35, 38, 39]
57
[ 0.  0. 55.  0.]
[40, 41, 44, 45]
58
[ 0.  4. 91.  8.]
[42, 43, 46, 47]
59
[ 0.  0.  0. 33.]


In [82]:
import sys
np.set_printoptions(threshold=sys.maxsize)

print(adjacent_m.shape)
print(adjacent_m[0:48, 48:60])

(60, 60)
[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [25.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 2.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0. 12.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. 18.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  4.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. 14.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  9.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  2.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. 36.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. 